In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import os
from collections import deque
from catboost import CatBoostRegressor

In [ ]:
cnt = 0
cap = cv2.VideoCapture("VID_20260412_174201.mp4")
while cap.isOpened():
    ret , frame = cap.read()
    if not ret:
        break
    if cnt % 10 == 0: # можно не 10 но я захотел 10
        cv2.imwrite(f"images/{cnt}678366.jpg", frame)
    cnt += 1
cap.release()

In [11]:
input_dir = 'images' # все по hsv
output_dir = 'labels'
LOWER_BLUE = np.array([85, 100, 100]) 
UPPER_BLUE = np.array([125, 255, 255])
for i in os.listdir(input_dir):
    img = cv2.imread(os.path.join(input_dir, i)) # склейка пути
    h, w, _ = img.shape # _ это заглушка 
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) 
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE) 
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) # ищет белый цвет
    coords = [] # для хранения координат как требует yolo
    for cnt in contours:    
        bx, by, bw, bh = cv2.boundingRect(cnt) # описывает контур в квадрат
        if cv2.contourArea(cnt) < 20: # убираем мелкий шум
            continue
        coords.append(f"0 {(bx+bw/2)/w:.6f} {(by+bh/2)/h} {bw/w} {bh/h}") # так требует yolo
    with open(os.path.join(output_dir, os.path.splitext(i)[0] + ".txt"), "w") as f: # генерит имя файла и делает доступным для записи
        f.write("\n".join(coords)) # сохраняет найденные координаты

In [7]:
model = YOLO('yolo26s.pt')
if __name__ == '__main__':
    model.train(
        data = r'C:\Users\user\projectolymp\data.yaml',
        epochs=100,
        imgsz=640, 
        batch=20,
        device="cuda" if torch.cuda.is_available() else "cpu", 
        project="models",
        name="1",
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        mosaic=1.0,
        mixup=0.0,
        copy_paste=0.0,
        degrees=15.0,
        translate=0.1,
        shear=0.0,      
        flipud=0.0, 
        fliplr=0.54,
        close_mosaic=10 
    )

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.36  Python-3.11.9 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\user\projectolymp\data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.54, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_

In [ ]:
model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\115\weights\best.pt')
cb_model = CatBoostRegressor()
cb_model.load_model("catboost.cbm")
cap = cv2.VideoCapture(0)
track_history = {}
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break
    results = model.track(frame, persist=True, conf=0.6) # ищет маркеры и присваевает им id
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history.get(track_id, deque(maxlen=150))
            track.append((float(x), float(y)))
            track_history[track_id] = track

            for i in range(1, len(track)):
                pt1 = (int(track[i-1][0]), int(track[i-1][1]))
                pt2 = (int(track[i][0]), int(track[i][1]))
                cv2.line(frame, pt1, pt2, (0, 0, 255), 2)


            if len(track) >= 10:
                last_10_points = list(track)[-10:]
                input_data = [coord for pt in last_10_points for coord in pt]
                prediction = cb_model.predict([input_data]) 
                

                pred_x, pred_y = prediction[0] 
                cv2.circle(frame, (int(pred_x), int(pred_y)), 7, (255, 0, 0), -1)


    cv2.imshow("MMC", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): # на q закрытие
        break
cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 55.4ms
Speed: 2.9ms preprocess, 55.4ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 markers, 5.1ms
Speed: 1.3ms preprocess, 5.1ms inference, 12.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 6.2ms
Speed: 1.5ms preprocess, 6.2ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 5.8ms
Speed: 1.1ms preprocess, 5.8ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 5.8ms
Speed: 1.2ms preprocess, 5.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 5.9ms
Speed: 0.9ms preprocess, 5.9ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 5.8ms
Speed: 1.3ms preprocess, 5.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 5.6ms
Speed: 0.9ms preprocess, 5.6ms inference, 0.5ms postprocess per image at shape (1, 3, 